In [1]:
import os
import subprocess, sys, time, warnings
from pathlib import Path

from adaptive_behaviour import paths

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

TRIALS  = paths.RUNNERS
SOURCE  = paths.RUNNERS / "trial_dpefe.py"
PATCHED = paths.RUNNERS_GENERATED / "trial_dpefe_saveB.py"
OUT     = paths.RESULTS_PLANNING_DEPTH; OUT.mkdir(parents=True, exist_ok=True)

HORIZON = 5        # DP-5, a mid-range planning horizon
SEEDS   = [0, 1, 2]   # DPEFE is expensive; a few seeds suffice for a structural claim
MAX_H   = 20       # evaluate predictive entropy out to the deepest horizon in the paper

assert SOURCE.exists(), f"published DPEFE runner not found at {SOURCE}"
print(f"horizon {HORIZON}, seeds {SEEDS}, predictive depth 1..{MAX_H}")

# Pin BLAS to one thread per worker. Each runner otherwise spawns one BLAS thread per core,
# so N concurrent runners create N x 12 threads on a 12-core machine. With 2432-state matrices
# threaded BLAS gives no speed-up, and the contention dominated everything on the first attempt
# (load average 55 on 12 cores).
BLAS_ENV = dict(OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1",
                VECLIB_MAXIMUM_THREADS="1", NUMEXPR_NUM_THREADS="1")


horizon 5, seeds [0, 1, 2], predictive depth 1..20


In [2]:
src = SOURCE.read_text()

anchor = "p_data.to_csv(file1, index=False)"
assert anchor in src, "PATCH FAILED: expected output line not found in trial_dpefe.py"

save_block = (
    anchor
    + "\n\n# --- added by 02_planning_depth.ipynb: persist the learned transition matrix ---\n"
      "import numpy as _np\n"
      "_np.save(paths.RESULTS_PLANNING_DEPTH / ('B_h%d_seed%d.npy' % (planning_horizon, mtrial)),\n"
      "         _np.array(dpefe_agent.B[0], dtype=float))\n"
)
out = src.replace(anchor, save_block, 1)

# Route behaviour CSVs away from the published directory. The published runner writes to
# results/published/dpefe/M<T>/, which is where the paper's per-seed data lives - writing there would
# destroy it.
redirect_from = "OUT_DIR = paths.results_dpefe_raw(planning_horizon)\nOUT_PREFIX = ''"
redirect_to   = ("OUT_DIR = paths.RESULTS_PLANNING_DEPTH\n"
                 "OUT_PREFIX = 'h' + str(planning_horizon) + '_'")
assert redirect_from in out, "PATCH FAILED: output path anchor not found"
out = out.replace(redirect_from, redirect_to)

# Hard guard: the patched runner must not reference any published data directory.
assert "results_dpefe_raw" not in out, "PATCH FAILED: patched runner still writes to a published directory"

PATCHED.write_text(out)
print(f"wrote {PATCHED.name}")
print("saves B[0] (ball-x factor, shape 38x38x3) per seed")

wrote trial_dpefe_saveB.py
saves B[0] (ball-x factor, shape 38x38x3) per seed


In [3]:
def b_path(seed):
    return OUT / f"B_h{HORIZON}_seed{seed}.npy"


def run(seed):
    t0 = time.time()
    r = subprocess.run([sys.executable, str(PATCHED), str(seed), str(HORIZON)],
                       capture_output=True, text=True, cwd=str(TRIALS),
                       env=dict(os.environ, **BLAS_ENV))
    return seed, time.time() - t0, r.returncode, r.stderr[-400:] if r.returncode else ""


pending = [s for s in SEEDS if not b_path(s).exists()]
print(f"{len(SEEDS) - len(pending)} already done; running {len(pending)}\n")

for i, s in enumerate(pending, 1):
    seed, dt, rc, err = run(s)
    print(f"[{i}/{len(pending)}] seed {seed}: {'OK' if rc == 0 else 'FAILED'} in {dt/60:.1f} min")
    if rc:
        print(err); break
    if i == 1 and len(pending) > 1:
        print(f"    -> estimated remaining: {(len(pending)-1) * dt/60:.0f} min")

available = [s for s in SEEDS if b_path(s).exists()]
print(f"\nmatrices available for seeds: {available}")

0 already done; running 3

[1/3] seed 0: OK in 231.7 min
    -> estimated remaining: 463 min
[2/3] seed 1: OK in 77.8 min
[3/3] seed 2: OK in 50.1 min

matrices available for seeds: [0, 1, 2]


In [4]:
assert available, "no saved matrices yet - run the cell above"

N = 38
UNIFORM = np.log(N)
rows = []

for seed in available:
    B = np.load(b_path(seed))                     # (next_state, state, action)
    for a in range(B.shape[2]):
        M = B[:, :, a]
        M = M / M.sum(axis=0, keepdims=True)      # columns are distributions over next state
        P = np.eye(N)
        for h in range(1, MAX_H + 1):
            P = M @ P
            col = P / P.sum(axis=0, keepdims=True)
            ent = -(col * np.log(np.clip(col, 1e-16, None))).sum(axis=0)
            rows.append(dict(seed=seed, action=a, h=h, nte=ent.mean() / UNIFORM))

depth = pd.DataFrame(rows)
curve = depth.groupby("h").nte.agg(["mean", "std"]).round(4)
print("Normalised entropy of the h-step-ahead prediction")
print("(1.0 = uniform, i.e. the rollout carries no information)\n")
print(curve.to_string())

Normalised entropy of the h-step-ahead prediction
(1.0 = uniform, i.e. the rollout carries no information)

      mean     std
h                 
1   0.3239  0.0538
2   0.5131  0.0660
3   0.6052  0.0712
4   0.6656  0.0748
5   0.7070  0.0779
6   0.7383  0.0804
7   0.7623  0.0828
8   0.7816  0.0848
9   0.7971  0.0867
10  0.8100  0.0885
11  0.8207  0.0902
12  0.8298  0.0917
13  0.8374  0.0932
14  0.8440  0.0946
15  0.8496  0.0959
16  0.8545  0.0972
17  0.8587  0.0984
18  0.8624  0.0995
19  0.8655  0.1007
20  0.8683  0.1017


In [5]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
m = depth.groupby("h").nte.mean()
s = depth.groupby("h").nte.std()
ax.plot(m.index, m.values, marker="o", color="#2F6F5E")
ax.fill_between(m.index, m - s, m + s, alpha=0.2, color="#2F6F5E")
ax.axhline(1.0, ls="--", lw=1, color="#A64B2A", label="uniform (no information)")
for T in (2, 5, 10, 20):
    ax.axvline(T, ls=":", lw=0.8, color="grey")
    ax.text(T, 0.02, f"DP-{T}", rotation=90, fontsize=7, va="bottom", ha="right", color="grey")
ax.set_xlabel("prediction depth $h$ (steps)")
ax.set_ylabel("normalised entropy of $\\mathbb{B}^h s$")
ax.set_ylim(0, 1.05); ax.legend(fontsize=8); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT / "predictive_entropy_vs_depth.png", dpi=200)
print(f"saved {paths.rel(OUT / 'predictive_entropy_vs_depth.png')}")
plt.show()

saved <repo>/main/pong_trials/planning_depth/predictive_entropy_vs_depth.png


In [ ]:
from adaptive_behaviour.pong.analysis.tables import planning_depth_table as P

# The analysis above is reproduced in planning_depth_table.py so that the notebook and the
# supplementary table cannot drift apart; the module recomputes the curve from the same saved
# matrices and emits the LaTeX directly.
s = P.write_table()
print("Normalised entropy of the h-step-ahead prediction (Supplementary Table S10)")
print(s[s.h.isin(P.REPORT_DEPTHS)].round(4).to_string(index=False))
print("\nwrote tables/tab_planning_depth.tex")